In [ ]:
import sys
if 'google.colab' in sys.modules:
    !uv pip install --system --pre ngsolve

# Scattering Problem

We consider scattering of an incident plane wave by an obstacle $\Omega\subset\mathbb R^3$:

\begin{align*}
\Delta u+\kappa^2u=0 \quad \text{in }\Omega^c,
\qquad u_{in}(x)=e^{i\kappa x_1}.
\end{align*}


The Brakhage-Werner formulation combines single and double layer operators.


The solution is represented as

$$
u = (i \kappa S - D) \phi,
$$

where $\phi$ solves the boundary integral equation
\begin{align*}
    \left( \tfrac{1}{2} + K + i \kappa V \right) \phi = u_{in} \qquad \text{on} \, \Gamma = \partial \Omega.
\end{align*}

Both layer operators use the Helmholtz fundamental solution

$$
G_\kappa(x,y) = \frac{e^{i\kappa \|x-y\|}}{4\pi\,\|x-y\|}.
$$

In [ ]:
from netgen.occ import *
from ngsolve import *
from ngsolve.webgui import Draw
from ngsolve.bem import *

In [ ]:
kappa=20
order=3

In [ ]:
screen = WorkPlane(Axes( (0,0,0), Z, X)).RectangleC(15,15).Face()

sphere = Sphere( (0,0,0), pi)
screen = screen - sphere
sp = Fuse(sphere.faces)
screen.faces.name="screen"
sp.faces.name="sphere"
shape = Compound([screen,sp])

mesh = shape.GenerateMesh(maxh=5/kappa).Curve(order)
Draw (mesh);

In [ ]:
fes_sphere = Compress(SurfaceL2(mesh, order=order, complex=True, definedon=mesh.Boundaries("sphere")))
u,v = fes_sphere.TnT()
fes_screen = Compress(SurfaceL2(mesh, order=order, dual_mapping=True, complex=True, definedon=mesh.Boundaries("screen")))
print ("ndof_sphere = ", fes_sphere.ndof, "ndof_screen =", fes_screen.ndof)

In [ ]:
fmm_params = {"fmm_maxdirect":50, "fmm_minorder":10, "fmm_order_factor":1.5, "fmm_separation":2.0}
with TaskManager():
    C = HelmholtzCF(u*ds("sphere"), kappa, **fmm_params) * v *ds("sphere")
    u,v  = fes_sphere.TnT()
    Id = BilinearForm(u*v*ds).Assemble()

In [ ]:
with TaskManager():
    lhs = 0.5 * Id.mat + C.mat

source = exp(1j * kappa * x) 
rhs = LinearForm(source*v*ds).Assemble()

In [ ]:
gfu = GridFunction(fes_sphere)
pre = BilinearForm(u*v*ds, diagonal=True).Assemble().mat.Inverse()
with TaskManager():
    gfu.vec[:] = solvers.GMRes(A=lhs, b=rhs.vec, pre=pre, maxsteps=80, tol=1e-8, printrates=False)

In [ ]:
Draw (gfu, order=5, min=-1, max=1);

## Postprocessing on screen

In [ ]:
uscat = GridFunction(fes_screen)
sl = HelmholtzSL(u*ds("sphere"),kappa, **fmm_params)
dl = HelmholtzDL(u*ds("sphere"), kappa, **fmm_params)
cf = 1j*kappa*sl - 1*dl

with TaskManager():
    cfgf = cf(gfu,mesh.Boundaries("screen"))
    uscat.Set(cfgf,definedon=mesh.Boundaries("screen"))

In [ ]:
print ("Scattered field")
Draw (uscat, mesh, min=-1,max=1, animate_complex=True, order=4);

In [ ]:
uin = mesh.BoundaryCF( {"screen": source }, default=0)
print ("Total field")
Draw (uin+uscat, mesh, min=-1,max=1, animate_complex=True, order=4);

## Head Simulation (HRTF)

Head Related Transfer Functions (HRTF) model how the human head and ears affect incoming sound. The same Brakhage-Werner combined field operator was used here for a more complex acoustic scattering geometry.

![Head geometry simulation](images/head.png)

Simulation details:
| quantity | value |
|---|---|
| mesh | 27202 vertices, 13600 second order triangles |
| space | complex `SurfaceL2`, order 2, 81600 dofs |
| frequency | $f=5000\,\mathrm{Hz}$, $c=343\,\mathrm{m/s}$, $\kappa=2\pi f/c/1000 = 0.0915916\,\mathrm{mm}^{-1}$ |
| integration points | 870,400 source and target points |
| multipoles | 18,830,910 coefficients, 301.29 MB |
| dense Galerkin matrix | $81600^2$ complex entries $\approx 106.5$ GB, about $354\times$ the multipole memory |
| Sauter-Schwab correction | $0.0959\%$ of all dof pairs - the remaining $99.9041\%$ use the FMM matrix action |
| solve | GMRes, 63 iterations, 108.99 s (on Mac M5 Pro) |